In [ ]:
import pandas as pd

In [ ]:
chunk_list = []

chunk_size = 10000 
json_reader = pd.read_json('windows_events.json', lines=True, chunksize=chunk_size)

for chunk in json_reader:
    chunk = chunk[['rule', 'agent', 'manager', 'id']] 
    chunk_list.append(chunk)

df = pd.concat(chunk_list, ignore_index=True)

print(df.head())
print("File loaded successfully without crashing!")

In [ ]:
print(df.shape)

In [ ]:
print("Total logs loaded:", len(df))

In [ ]:
df['Rule_Level'] = df['rule'].apply(lambda x: x.get('level') if isinstance(x, dict) else 0)
df['Rule_Description'] = df['rule'].apply(lambda x: x.get('description') if isinstance(x, dict) else 'None')

df['Agent_Name'] = df['agent'].apply(lambda x: x.get('name') if isinstance(x, dict) else 'Unknown')
df['Agent_IP'] = df['agent'].apply(lambda x: x.get('ip') if isinstance(x, dict) else 'Unknown')

print(df[['Rule_Level', 'Rule_Description', 'Agent_Name', 'Agent_IP']].head())

In [ ]:
df_clean = df.drop(columns=['rule', 'agent', 'manager'])

df_clean = df_clean.dropna(subset=['Rule_Level'])

print(df_clean.shape)

In [ ]:
df_clean['Is_High_Severity'] = df_clean['Rule_Level'].apply(
    lambda x: 1 if x >= 10 else 0
)

df_clean['Is_Login_Failure'] = df_clean['Rule_Description'].apply(lambda x: 1 if 'Logon Failure' in str(x) else 0)

df_clean = df_clean.fillna(0)

In [ ]:
from sklearn.model_selection import train_test_split

features = ['Rule_Level', 'Is_High_Severity', 'Is_Login_Failure']
X = df_clean[features]

X_train, X_test = train_test_split(X, test_size=0.3, random_state=42)

print("Training data size:", len(X_train))
print("Testing data size:", len(X_test))

In [ ]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(contamination=0.01, random_state=42)

model.fit(X_train)
print("Model training complete!")

In [ ]:
predictions = model.predict(X_test)

X_test_results = X_test.copy()
X_test_results['Anomaly_Score'] = predictions

anomalies = X_test_results[X_test_results['Anomaly_Score'] == -1]
print("Total Anomalies Found:", len(anomalies))
print(anomalies.head())

In [ ]:
anomalies.to_csv('Detected_Threats.csv', index=False)
print("Threat report saved successfully!")

In [ ]:
df_clean.head(10000).to_csv('clean_logs.csv', index=False)